# Efficient & long-context attention (Longformer, BigBird, FlashAttention) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Keep useful attention links while avoiding materializing the full quadratic matrix
The examples count dense cost, local windows, global tokens, random BigBird links and streaming softmax equivalence.

In [ ]:
T=np.array([128,256,512,1024]); dense=T*T
plt.figure(figsize=(6,3)); plt.plot(T,dense,'-o'); plt.title('Dense attention memory grows as T squared')
assert dense[-1]/dense[0]==64

In [ ]:
T=32; w=2; mask=np.fromfunction(lambda i,j: abs(i-j)<=w,(T,T));
plt.figure(figsize=(4,3)); plt.imshow(mask,cmap='Greens'); plt.title('Longformer local attention window')
assert mask.sum()<T*T and mask[10,12]

In [ ]:
T=20; w=1; mask=np.fromfunction(lambda i,j: abs(i-j)<=w,(T,T)); mask[:,0]=True; mask[0,:]=True
plt.figure(figsize=(4,3)); plt.imshow(mask,cmap='Blues'); plt.title('Global token connects across the sequence')
assert mask[19,0] and mask[0,19]

In [ ]:
rng=np.random.default_rng(0); T=24; links=np.zeros((T,T),bool)
for i in range(T): links[i,rng.choice(T,3,replace=False)]=True
plt.figure(figsize=(4,3)); plt.imshow(links,cmap='Oranges'); plt.title('BigBird-style random sparse links')
assert links.sum()==72

In [ ]:
x=np.array([10.,9.,8.,1.]); full=softmax(x)
chunks=[x[:2],x[2:]]; m=max(c.max() for c in chunks); denom=sum(np.exp(c-m).sum() for c in chunks); stream=np.concatenate([np.exp(c-m)/denom for c in chunks])
plt.figure(figsize=(5,3)); plt.bar(range(4),stream); plt.title('FlashAttention streams exact softmax without full matrix')
assert np.allclose(full,stream)